# 🐱 MEL WAN LoRA Training
全自動訓練 WAN 2.2 專用 MEL 角色 LoRA

**硬體**: T4 16GB+ | **時間**: ~2小時 | **輸出**: .safetensors

In [ ]:
# @title 1. 環境設置 & 下載資料
import os, sys, tarfile, urllib.request
from pathlib import Path

# GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# ── Download lora_trainer (12MB) ──
print("📥 lora_trainer (12MB)...")
URL = "https://raw.githubusercontent.com/melandz/mel-wan-lora-colab/main/lora_trainer.tar.gz"
urllib.request.urlretrieve(URL, "/content/lora_trainer.tar.gz")
with tarfile.open("/content/lora_trainer.tar.gz") as tf:
    tf.extractall("/content/")
assert Path("/content/lora_trainer/anima_train_network.py").exists(), "❌ lora_trainer failed!"
print("✅ lora_trainer OK")

# ── Download dataset (16MB) ──
print("📥 mel_wan_dataset (16MB)...")
URL2 = "https://raw.githubusercontent.com/melandz/mel-wan-lora-colab/main/mel_wan_dataset.tar.gz"
urllib.request.urlretrieve(URL2, "/content/mel_wan_dataset.tar.gz")
with tarfile.open("/content/mel_wan_dataset.tar.gz") as tf:
    tf.extractall("/content/dataset/")
n_img = len(list(Path("/content/dataset/images").glob("*")))
n_cap = len(list(Path("/content/dataset/captions").glob("*")))
assert n_img >= 20, f"❌ only {n_img} images!"
print(f"✅ Dataset OK: {n_img} images, {n_cap} captions")

# ── Install deps (transformers pinned for CLIPFeatureExtractor compatibility) ──
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q accelerate sentencepiece protobuf einops huggingface_hub safetensors peft deepspeed bitsandbytes
!pip install -q "transformers==4.36.2"

print("✅ Environment ready!")

In [ ]:
# @title 2. 下載 WAN 2.2 模型 (~28GB, 10-20分)
from pathlib import Path

MODEL_DIR = Path('/content/wan22_model')
QWEN3_DIR = Path('/content/qwen3_06b')

print("📥 WAN 2.2 DiT (~28GB)...")
!huggingface-cli download Wan-AI/Wan2.2-I2V-A14B --local-dir {MODEL_DIR} \
  --include "diffusion_pytorch_model*.safetensors" 2>&1 | tail -5

print("📥 Qwen3 0.6B...")
!huggingface-cli download Qwen/Qwen3-0.6B --local-dir {QWEN3_DIR} 2>&1 | tail -3

print("✅ Models ready")
!ls -lh {MODEL_DIR}/ | head -8

In [ ]:
# @title 3. 準備訓練資料
import json
from pathlib import Path

IMG_DIR = Path('/content/dataset/images')
CAP_DIR = Path('/content/dataset/captions')

metadata = []
for img in sorted(IMG_DIR.glob('*')):
    if img.suffix.lower() not in ['.png','.jpg','.jpeg']:
        continue
    cap_path = CAP_DIR / (img.stem + '.txt')
    if not cap_path.exists():
        for c in CAP_DIR.glob('*.txt'):
            if img.stem.split('_',1)[-1] in c.stem:
                cap_path = c; break
    caption = cap_path.read_text().strip() if cap_path.exists() else "mel, silver hair, purple eyes, cat ears"
    metadata.append({"image": str(img), "caption": caption})

print(f"📊 {len(metadata)} images")
for m in metadata[:3]:
    print(f"  {Path(m['image']).name}: {m['caption'][:50]}...")

with open('/content/dataset.json','w') as f:
    for m in metadata:
        f.write(json.dumps(m,ensure_ascii=False)+'\n')
print("✅ dataset.json saved")

In [ ]:
# @title 4. 建立訓練 Config
from pathlib import Path

config = f"""pretrained_model_name_or_path = "/content/wan22_model/diffusion_pytorch_model.safetensors"
qwen3 = "/content/qwen3_06b"
vae = "/content/wan22_model/vae"

output_dir = "/content/drive/MyDrive/mel_wan_lora_output"
output_name = "mel_wan_lora_v1"

dataset_config = "/content/dataset.json"

resolution = "832,480"
batch_size = 1
max_train_epochs = 4
save_every_n_epochs = 1

network_dim = 8
network_alpha = 16

learning_rate = 1e-4
lr_scheduler = "cosine"
lr_warmup_steps = 50
optimizer_type = "adamw8bit"

mixed_precision = "bf16"
gradient_checkpointing = true
blocks_to_swap = 28
cpu_offload_checkpointing = true

cache_latents = true
cache_text_encoder_outputs = true

attn_mode = "torch"
split_attn = false

discrete_flow_shift = 5.0
weighting_scheme = "none"
logit_mean = 0.0
logit_std = 1.0
mode_scale = 1.0

log_with = "tensorboard"
logging_dir = "/content/logs"

sample_prompts = "/content/sample_prompts.txt"
"""
Path('/content/train_config.toml').write_text(config)
Path('/content/sample_prompts.txt').write_text(
    '{"prompt": "mel, silver hair, purple eyes, cat ears, school uniform, rooftop sunset, dancing", "negative_prompt": "worst quality, blurry"}\n'
)
print("✅ config + prompts ready")

In [ ]:
# @title 5. 🔥 開始訓練
import sys, os
sys.path.insert(0, '/content/lora_trainer')
os.chdir('/content/lora_trainer')

!python /content/lora_trainer/anima_train_network.py \
  --config_file /content/train_config.toml \
  2>&1 | tee /content/train_log.txt

print("\n✅ Training complete!")
print("📁 Output: /content/drive/MyDrive/mel_wan_lora_output/")

## 📝 注意
- 全部自動下載，不需手動上傳
- WAN 2.2 模型 28GB 下載需 10-20 分鐘
- LoRA 存到 Google Drive → 下載 `.safetensors` 丟 ComfyUI